# Function 5 - Week 3: model predictions

Compare models using leave-one-out validation, including the latest, highest output. Use the fitted Gaussian process and an acquisition function to propose the next submission.

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import LeaveOneOut, cross_val_predict
import sys

shared_dir = Path.cwd().resolve().parents[1] / "Shared"
if str(shared_dir) not in sys.path:
    sys.path.insert(0, str(shared_dir))
from coverage_helpers import make_coverage_space
from coverage_cache import compare_coverage_batches
from query_budget import remaining_queries, TOTAL_QUERY_WEEKS


In [ ]:
# Data and output paths for this week's notebook.
# Start the kernel with this notebook's folder as its working directory.
week_dir = Path.cwd().resolve()
full_data_path = week_dir / "observations.csv"

if not (week_dir / "model_predictions.ipynb").is_file() or not full_data_path.is_file():
    raise FileNotFoundError(
        "Run this notebook with its own folder as the working directory; "
        "model_predictions.ipynb and observations.csv must be together."
    )

figures = week_dir / "figures"
figures.mkdir(parents=True, exist_ok=True)

In [ ]:
# Basic data load
print(f"Loading data from: {full_data_path}")
data = pd.read_csv(full_data_path)

In [ ]:
# Column inspection
columns = data.columns
print("Columns: ", columns)

In [ ]:
# Where we pray for good data, aka checking for nulls
print (data.isna().sum())

In [ ]:
# Splitting our data into inputs X and outputs y
output_column = "y"
# The course names inputs x1, x2, ...; preserve their CSV column order.
input_columns = [
    name for name in data.columns
    if name.startswith("x") and name[1:].isdigit()
]

#print("Input columns: ", input_columns)
#print("Output column: ", output_column)

X = data[input_columns]
y = data[output_column]

#print("The inputs are given by :")
#print(X.head())
#print("The outputs are given by :")
#print(y[:5])


In [ ]:
# Set up leave-one-out validation for the model comparisons that follow.
cross_validation = LeaveOneOut()
number_of_folds = cross_validation.get_n_splits(X)

print(f"Validation folds: {number_of_folds}")
print(f"Training observations per fold: {len(X) - 1}")
print("Validation observations per fold: 1")

In [ ]:
# Identify the one new observation in this week's accumulated data.
latest_round = int(data["source_round"].max())
latest_indices = data.index[data["source_round"].eq(latest_round)]
if latest_round == 0 or len(latest_indices) != 1:
    raise ValueError("This weekly comparison expects one latest returned observation.")
latest_index = latest_indices[0]
latest_position = X.index.get_loc(latest_index)
latest_output = float(y.loc[latest_index])
other_observations = data["source_round"].lt(latest_round).to_numpy()

print(f"Latest observation: {data.loc[latest_index, 'observation_id']}")
print(f"Latest output: {latest_output:.6f}")
print(f"Previous best: {y.loc[data['source_round'].lt(latest_round)].max():.6f}")
if y.gt(0).all():
    print(f"Output magnitudes span {np.log10(y.max() / y.min()):.2f} orders of magnitude.")
data.loc[[latest_index], ["observation_id", *input_columns, output_column]]

In [ ]:
# Establish a mean-prediction benchmark using leave-one-out validation.
baseline_model = DummyRegressor(strategy="mean")

baseline_predictions = cross_val_predict(
    baseline_model,
    X,
    y,
    cv=cross_validation,
)

baseline_rmse = root_mean_squared_error(y, baseline_predictions)

print(f"Baseline RMSE: {baseline_rmse:.4f}")

In [ ]:
# Evaluate whether linear input effects improve on the mean baseline.
linear_model = LinearRegression()

linear_predictions = cross_val_predict(
    linear_model,
    X,
    y,
    cv=cross_validation,
)

linear_rmse = root_mean_squared_error(y, linear_predictions)
linear_improvement = 1 - linear_rmse / baseline_rmse

print(f"Mean baseline RMSE: {baseline_rmse:.4f}")
print(f"Linear regression RMSE: {linear_rmse:.4f}")
print(f"RMSE reduction against baseline: {linear_improvement:.1%}")

In [ ]:
# Evaluate linear regression with a penalty on large coefficients.
ridge_model = Ridge(alpha=1.0)

ridge_predictions = cross_val_predict(
    ridge_model,
    X,
    y,
    cv=cross_validation,
)

ridge_rmse = root_mean_squared_error(y, ridge_predictions)
ridge_improvement = 1 - ridge_rmse / baseline_rmse

print(f"Ridge regression RMSE: {ridge_rmse:.4f}")
print(f"RMSE reduction against baseline: {ridge_improvement:.1%}")

In [ ]:
# Evaluate a model that can capture curved input-output relationships.
kernel_ridge_model = KernelRidge(
    alpha=0.1,
    kernel="rbf",
    gamma=1.0,
)

kernel_ridge_predictions = cross_val_predict(
    kernel_ridge_model,
    X,
    y,
    cv=cross_validation,
)

kernel_ridge_rmse = root_mean_squared_error(y, kernel_ridge_predictions)
kernel_ridge_improvement = 1 - kernel_ridge_rmse / baseline_rmse

print(f"Kernel ridge regression RMSE: {kernel_ridge_rmse:.4f}")
print(f"RMSE reduction against baseline: {kernel_ridge_improvement:.1%}")

In [ ]:
# Evaluate a Gaussian process on the original output scale, as in Function 4.
# Normalisation and kernel fitting happen inside each training fold.
# Assume noise SD is 10% of training output SD; the true noise level is unknown.
noise_fraction = 0.10

gaussian_process_model = GaussianProcessRegressor(
    kernel=ConstantKernel(1.0, (0.01, 100.0)) * RBF(
        length_scale=0.3,
        length_scale_bounds=(0.03, 2.0),
    ),
    alpha=noise_fraction ** 2,
    normalize_y=True,
    n_restarts_optimizer=3,
    random_state=42,
)
gaussian_process_predictions = cross_val_predict(
    gaussian_process_model, X, y, cv=cross_validation,
)
gaussian_process_rmse = root_mean_squared_error(y, gaussian_process_predictions)
print(f"Gaussian-process RMSE: {gaussian_process_rmse:.4f}")
print(f"RMSE reduction against baseline: {1 - gaussian_process_rmse / baseline_rmse:.1%}")

In [ ]:
# Check whether modelling logarithmic outputs handles their wide range better.
# The wrapper applies log during each fit and exp when returning predictions.
# exp(log-space mean) is a posterior median in original units, not its mean.
# Here the assumed noise fraction applies to training log-output SD.
if not y.gt(0).all():
    raise ValueError("Natural-log targets require strictly positive outputs.")

log_gaussian_process_model = TransformedTargetRegressor(
    regressor=clone(gaussian_process_model),
    func=np.log,
    inverse_func=np.exp,
)
log_gaussian_process_predictions = cross_val_predict(
    log_gaussian_process_model, X, y, cv=cross_validation,
)
log_gaussian_process_rmse = root_mean_squared_error(y, log_gaussian_process_predictions)
print(f"Log-target GP RMSE in original output units: {log_gaussian_process_rmse:.4f}")
print(f"RMSE reduction against baseline: {1 - log_gaussian_process_rmse / baseline_rmse:.1%}")

In [ ]:
# Compare every model in the same original output units, retaining all observations.
models = {
    "Mean baseline": baseline_model,
    "Linear regression": linear_model,
    "Ridge": ridge_model,
    "Kernel ridge": kernel_ridge_model,
    "Gaussian process": gaussian_process_model,
    "GP with log target (median)": log_gaussian_process_model,
}
held_out_predictions = {
    "Mean baseline": baseline_predictions,
    "Linear regression": linear_predictions,
    "Ridge": ridge_predictions,
    "Kernel ridge": kernel_ridge_predictions,
    "Gaussian process": gaussian_process_predictions,
    "GP with log target (median)": log_gaussian_process_predictions,
}
comparison_rows = []
for name, predictions in held_out_predictions.items():
    error = root_mean_squared_error(y, predictions)
    comparison_rows.append({
        "Model": name,
        "LOO RMSE": error,
        "LOO MAE": mean_absolute_error(y, predictions),
        "RMSE reduction (%)": 100 * (1 - error / baseline_rmse),
    })
model_comparison = pd.DataFrame(comparison_rows).sort_values("LOO RMSE").reset_index(drop=True)
print("RMSE gives large errors more weight; MAE reports average absolute error.")
print(f"All models use the same {len(y)} leave-one-out folds and all observations are retained.")
model_comparison.round(4)

In [ ]:
# Inspect the latest-result fold: train on earlier observations and predict the new one.
# Model settings were chosen for this analysis, so this is a retrospective check.
# The other-row metric below uses folds that include the latest observation in training.
latest_rows = []
for name, predictions in held_out_predictions.items():
    residuals = predictions - y.to_numpy()
    latest_prediction = float(predictions[latest_position])
    latest_error = latest_prediction - latest_output
    latest_rows.append({
        "Model": name,
        "Latest held-out prediction": latest_prediction,
        "Latest error": latest_error,
        "Latest absolute error (%)": 100 * abs(latest_error) / abs(latest_output),
        "Latest share of squared error (%)": 100 * latest_error ** 2 / np.sum(residuals ** 2),
        "Other rows LOO RMSE": root_mean_squared_error(
            y.to_numpy()[other_observations], predictions[other_observations],
        ),
    })
latest_prediction_review = pd.DataFrame(latest_rows).sort_values("Latest absolute error (%)").reset_index(drop=True)
print(f"Actual latest output: {latest_output:.6f}")
print(f"Each prediction of this latest output used only the {other_observations.sum()} earlier observations.")
print("Other rows LOO RMSE excludes this row from scoring, not from their training folds.")
latest_prediction_review.round(4)

In [ ]:
# Plot the strongest nonlinear comparisons on identical axes, with an earlier-row zoom.
plot_models = ["Kernel ridge", "Gaussian process", "GP with log target (median)"]
full_values = np.concatenate([y.to_numpy(), *(held_out_predictions[name] for name in plot_models)])
zoom_values = np.concatenate([
    y.to_numpy()[other_observations],
    *(held_out_predictions[name][other_observations] for name in plot_models),
])
full_padding = 0.05 * np.ptp(full_values)
zoom_padding = 0.05 * np.ptp(zoom_values)
full_limits = (float(full_values.min() - full_padding), float(full_values.max() + full_padding))
zoom_limits = (float(zoom_values.min() - zoom_padding), float(zoom_values.max() + zoom_padding))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for column, name in enumerate(plot_models):
    predictions = held_out_predictions[name]
    for row, limits, observed_values, predicted_values in [
        (0, full_limits, y.to_numpy(), predictions),
        (1, zoom_limits, y.to_numpy()[other_observations], predictions[other_observations]),
    ]:
        ax = axes[row, column]
        ax.scatter(observed_values, predicted_values, alpha=0.75)
        ax.plot(limits, limits, "k--", linewidth=1, label="Perfect prediction")
        ax.set(xlim=limits, ylim=limits, xlabel="Observed output")
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)
    axes[0, column].scatter(
        latest_output, predictions[latest_position], marker="*", s=180,
        color="darkorange", edgecolor="black", label="Latest observation",
    )
    model_error = root_mean_squared_error(y, predictions)
    axes[0, column].set_title(f"{name}\nAll observations: RMSE {model_error:.1f}")
    axes[0, column].legend(fontsize=8, loc="upper left")
    axes[1, column].set_title("Earlier observations: zoom")
axes[0, 0].set_ylabel("Held-out predicted output")
axes[1, 0].set_ylabel("Held-out predicted output")
fig.suptitle("Function 5 - Week 3: leave-one-out model comparison", fontsize=15)
fig.text(
    0.5, 0.015,
    "The lower row zooms the earlier observations; their training folds include the latest output.\n"
    "All reported overall errors retain every observation. Log-target predictions are back-transformed medians.",
    ha="center", fontsize=10,
)
fig.tight_layout(rect=(0, 0.065, 1, 0.955))
fig.savefig(figures / "model-comparison-validation.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Compare each model's held-out prediction with the latest output.
latest_chart = latest_prediction_review.sort_values("Latest held-out prediction", ascending=False)
fig, ax = plt.subplots(figsize=(11, 5.5))
positions = np.arange(len(latest_chart))
values = latest_chart["Latest held-out prediction"].to_numpy()
ax.barh(positions, values, color="#3279a8")
ax.set_yticks(positions, latest_chart["Model"])
ax.invert_yaxis()
ax.axvline(latest_output, color="darkorange", linewidth=2, linestyle="--", label=f"Observed: {latest_output:.2f}")
chart_max = max(latest_output, values.max())
chart_min = min(0.0, values.min())
chart_padding = 0.03 * (chart_max - chart_min)
for position, value in zip(positions, values):
    ax.text(value + chart_padding, position, f"{value:.1f}", va="center", fontsize=10)
ax.set_xlim(chart_min - chart_padding, chart_max + 4 * chart_padding)
ax.set_xlabel("Predicted output at the latest query, with its result held out")
ax.set_title(f"Function 5 - Week 3: predicting the new best output\nTrain on {other_observations.sum()} earlier observations; actual output {latest_output:.2f}")
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(figures / "latest-output-predictions.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Check fitting with the new result included, separately from held-out performance.
# These fitted values are training predictions and are not validation evidence.
fitted_models = {}
fit_review_rows = []
for name, model in models.items():
    fitted_model = clone(model).fit(X, y)
    fitted_models[name] = fitted_model
    fitted_latest = float(fitted_model.predict(X.loc[[latest_index]])[0])
    fit_review_rows.append({
        "Model": name,
        "Observed latest output": latest_output,
        "Held-out prediction": held_out_predictions[name][latest_position],
        "Fitted prediction (training)": fitted_latest,
    })
print("Including the new output may let a model fit it closely; this does not mean it predicted it in advance.")
pd.DataFrame(fit_review_rows).round(4)

In [ ]:
# Summarise the evidence before moving on to query selection.
lowest_rmse = model_comparison.iloc[0]
lowest_mae = model_comparison.loc[model_comparison["LOO MAE"].idxmin()]
closest_latest = latest_prediction_review.iloc[0]
lowest_rmse_latest = latest_prediction_review.loc[
    latest_prediction_review["Model"].eq(lowest_rmse["Model"])
].iloc[0]

print(f"Lowest overall LOO RMSE: {lowest_rmse['Model']} ({lowest_rmse['LOO RMSE']:.4f}).")
print(f"RMSE reduction against mean baseline: {lowest_rmse['RMSE reduction (%)']:.1f}%.")
print(f"Lowest overall LOO MAE: {lowest_mae['Model']} ({lowest_mae['LOO MAE']:.4f}).")
print(
    f"Closest prediction of the latest held-out result: {closest_latest['Model']}, "
    f"{closest_latest['Latest held-out prediction']:.4f} versus actual {latest_output:.4f}."
)
print(f"Its absolute error on that result: {closest_latest['Latest absolute error (%)']:.1f}%.")
print(
    f"The latest result contributes {lowest_rmse_latest['Latest share of squared error (%)']:.1f}% "
    "of the lowest-RMSE model's total held-out squared error."
)
print("Overall prediction error and ability to locate a better maximum are different assessments.")
print("This comparison does not establish one peak, two peaks or the location of the global maximum.")
print("Model comparison is complete; query selection follows below.")

In [ ]:
# Select the raw-output Gaussian process already fitted on all observations.
# Retain the Week 2 model choice; review the updated validation results above.
selected_model = fitted_models["Gaussian process"]

print(f"Selected model: {type(selected_model).__name__}")
print(f"Observations used for the final fit: {len(X)}")
print(f"Fitted kernel: {selected_model.kernel_}")
print(f"Assumed noise SD: {noise_fraction * y.std(ddof=0):.4f} output units")
print("Review the held-out errors above when assessing this model choice.")

In [ ]:
# Build the same broad Sobol candidate space used for Function 4.
candidate_inputs = pd.DataFrame(
    make_coverage_space(X.shape[1], power=14, seed=42),
    columns=input_columns,
)

# Function 5's best x3 and x4 are at the upper boundary.
# Also search that slice explicitly: interior Sobol points do not sample the face.
slice_inputs = candidate_inputs.copy()
slice_inputs[["x3", "x4"]] = X.loc[y.idxmax(), ["x3", "x4"]].to_numpy()

# Reuse the separate one-query coverage result, rather than a point from the five-query batch.
single_coverage_comparison = compare_coverage_batches(
    existing=X,
    cache_dir=week_dir / ".coverage-cache",
    max_queries=1,
)
single_coverage_inputs = pd.DataFrame(
    single_coverage_comparison["proposed_batches"][1],
    columns=input_columns,
)

candidate_inputs = pd.concat([
    candidate_inputs.assign(candidate_source="Full input space"),
    slice_inputs.assign(candidate_source="Best x3/x4 slice"),
    single_coverage_inputs.assign(candidate_source="Single coverage query"),
], ignore_index=True).drop_duplicates(subset=input_columns).reset_index(drop=True)

print(f"Candidate locations before excluding observed inputs: {len(candidate_inputs):,}")

In [ ]:
# Rank candidates by the same UCB rule used for Function 4.
# UCB = posterior mean + exploration weight * posterior standard deviation.
# The acquisition score is not a predicted output.
exploration_weight = 1.96
observed_coordinates = set(map(tuple, X.round(6).to_numpy()))
is_new_candidate = [
    tuple(row) not in observed_coordinates
    for row in candidate_inputs[input_columns].round(6).to_numpy()
]
bayesian_candidates = candidate_inputs.loc[is_new_candidate].copy().reset_index(drop=True)

candidate_mean, candidate_uncertainty = selected_model.predict(
    bayesian_candidates[input_columns], return_std=True,
)
bayesian_candidates["posterior_mean"] = candidate_mean
bayesian_candidates["posterior_std"] = candidate_uncertainty
bayesian_candidates["ucb"] = candidate_mean + exploration_weight * candidate_uncertainty
bayesian_candidates = bayesian_candidates.sort_values("ucb", ascending=False).reset_index(drop=True)
recommended_candidate = bayesian_candidates.iloc[0]

# Show the strongest candidate from each route, including pure coverage.
candidate_comparison = bayesian_candidates.loc[
    bayesian_candidates.groupby("candidate_source")["ucb"].idxmax()
].sort_values("ucb", ascending=False)
print(f"Unobserved candidate locations: {len(bayesian_candidates):,}")
print(f"Best observed output: {y.max():.6f}")
print(f"Exploration weight: {exploration_weight}")
print("Best individual alternative from each candidate source:")
candidate_comparison.round(6)

In [ ]:
# Summarise one proposed query and check sensitivity to the exploration weight.
# The maximum is over our sampled candidates, not a proven global maximum.
next_query = recommended_candidate[input_columns].astype(float).round(6)
next_query_inputs = pd.DataFrame([next_query.to_numpy()], columns=input_columns)
next_mean, next_std = selected_model.predict(next_query_inputs, return_std=True)

if not np.isfinite(next_query.to_numpy()).all() or not next_query.between(0, 0.999999).all():
    raise ValueError("The proposed query is outside the allowed input space.")
if tuple(next_query) in observed_coordinates:
    raise ValueError("The proposed query has already been observed.")

print("Proposed next query (input-column order):")
print(", ".join(f"{name}={value:.6f}" for name, value in next_query.items()))
print("Portal input: " + "-".join(f"{value:.6f}" for value in next_query))
print(f"Candidate source: {recommended_candidate['candidate_source']}")
print("Reason: highest sampled UCB score, compared with both the full-space and single-coverage alternatives.")
print(f"Best observed output: {y.max():.6f}")
print(f"Predicted output at the proposed query: {next_mean[0]:.6f}")
print(f"Posterior standard deviation: {next_std[0]:.6f}")
print(f"Predicted improvement from the mean: {next_mean[0] - y.max():.6f}")
print(f"UCB acquisition score: {next_mean[0] + exploration_weight * next_std[0]:.6f}")
print("The proposal remains conditional on the model assumptions; review the held-out errors above.")
print("This is a model-based proposal, not a guaranteed improvement or a submission.")

# Check noise assumptions at this same proposed location; do not refit selected_model.
# This is a sensitivity check, not a search for the best noise setting.
noise_sensitivity = []
for assumed_noise_fraction in [0.03, noise_fraction, 0.30]:
    if assumed_noise_fraction == noise_fraction:
        sensitivity_model = selected_model
    else:
        sensitivity_model = clone(selected_model).set_params(alpha=assumed_noise_fraction ** 2)
        sensitivity_model.fit(X, y)
    sensitivity_mean, sensitivity_std = sensitivity_model.predict(next_query_inputs, return_std=True)
    noise_sensitivity.append({
        "Assumed noise fraction": assumed_noise_fraction,
        "Predicted output": sensitivity_mean[0],
        "Posterior SD": sensitivity_std[0],
        "Predicted mean improvement": sensitivity_mean[0] - y.max(),
    })

print("Predictions at the proposed query under different noise assumptions:")
print(pd.DataFrame(noise_sensitivity).round(6).to_string(index=False))
print("The noise fraction is an assumption; these checks do not establish the true noise level.")

exploration_sensitivity = []
for weight in [0.0, 1.0, 1.96, 3.0]:
    scores = bayesian_candidates["posterior_mean"] + weight * bayesian_candidates["posterior_std"]
    candidate = bayesian_candidates.loc[scores.idxmax()]
    exploration_sensitivity.append({
        "Exploration weight": weight,
        **candidate[input_columns].to_dict(),
        "Predicted output": candidate["posterior_mean"],
        "Posterior SD": candidate["posterior_std"],
    })

print("Selected locations under different exploration weights (0 means pure exploitation):")
pd.DataFrame(exploration_sensitivity).round(6)

## Evidence for model-led queries

Use **GP RMSE / mean-baseline RMSE** as the main comparison: below 1 means better
held-out predictions than the baseline; 0.2 means one fifth of its error. The
existing model table reports the equivalent percentage reduction. This measures
predictive usefulness on the observed data, not the fraction of the input space explored.

Also check how many different locations the tested exploration weights select.
One location means this choice is stable across those weights for this candidate
sample and fitted model. Several locations mean the exploration trade-off still
changes the decision. Retain the noise-sensitivity check above.

The query budget provides context, not a confidence score. These checks do not
establish that exploration is complete or that the global optimum has been found;
GP uncertainty depends on its [kernel and noise assumptions](https://scikit-learn.org/stable/modules/gaussian_process.html).


In [ ]:
# Reuse the validation and sensitivity results already calculated above: no new fits.
relative_gp_error = gaussian_process_rmse / baseline_rmse if baseline_rmse > 0 else None
weight_choices = pd.DataFrame(exploration_sensitivity)
distinct_choices = len(weight_choices[input_columns].round(6).drop_duplicates())

model_evidence = pd.DataFrame([
    {
        "Check": "GP / mean-baseline held-out RMSE",
        "Result": f"{relative_gp_error:.3f} (1.000 = baseline)" if relative_gp_error is not None else "Undefined: baseline error is zero",
    },
    {
        "Check": "RMSE in output units: GP / baseline",
        "Result": f"{gaussian_process_rmse:.4f} / {baseline_rmse:.4f}",
    },
    {
        "Check": "Distinct proposed locations across exploration weights",
        "Result": f"{distinct_choices} across {len(weight_choices)} tested weights (six-decimal inputs)",
    },
    {
        "Check": "Queries available from the start of this week",
        "Result": f"{remaining_queries(week_dir)} of the {TOTAL_QUERY_WEEKS}-query course allowance",
    },
])
model_evidence
